## Build Results Fact
1. Read silver results table
2. Read silver sprints table
3. Add new column session_type with values RACE or SPRINT
4. UNION results and sprints
5. Derive additional columns
- is_win -> Indicates that the driver own the race
- is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
- has_points -> Indicates that the driver has scored points
6.Write the transformed data to gold fact_session_results table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00_common/01_environment_config

In [0]:
%run "../00_common/04_gold_helpers"

In [0]:
from pyspark.sql import functions as f

## Step 1 - Read source tables
- silver.results
- silver.sprints

In [0]:
results_silver_path = "s3://bizf1project123/Silver/Kalashri_increment/results"

sprints_silver_path = "s3://bizf1project123/Silver/Kalashri_increment/sprints"

target_path = "s3://bizf1project123/Gold/Kalashri_increment/fact_session_results"

In [0]:
results_df = (
    spark.read.format("delta")
        .load(results_silver_path)
        .filter(F.col("batch_id") == v_batch_id.strip())
        .withColumn("session_type", F.lit("RACE"))
        .drop(
            "race_name", "race_date", "ingestion_timestamp",
            "source_file", "batch_id", "created_timestamp",
            "updated_timestamp"
        )
)

display(results_df)

In [0]:
sprints_df = (
    spark.read.format("delta")
        .load(sprints_silver_path)
        .filter(F.col("batch_id") == v_batch_id.strip())
        .withColumn("session_type", F.lit("SPRINT"))
        .drop(
            "race_name", "race_date", "ingestion_timestamp",
            "source_file", "batch_id", "created_timestamp",
            "updated_timestamp"
        )
)

display(sprints_df)

## Step 2 - UNION results and sprints

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)
display(results_sprints_df)

## Step 3 - Add dervied columns
- is_win -> Indicates that the driver own the race
- is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
- has_points -> Indicates that the driver has scored points

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.filter("season = 2025"))

## Step 4 - Write the transformed data to the gold fact_session_results table

In [0]:
write_to_gold(
    input_df=fact_session_results_df,
    target_path=target_path,
    merge_condition="""
        t.season = s.season
        AND t.round = s.round
        AND t.constructor_id = s.constructor_id
        AND t.driver_id = s.driver_id
        AND t.session_type = s.session_type
    """,
    columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]
)

In [0]:
%sql
SELECT *
FROM delta.`s3://bizf1project123/Gold/Kalashri_increment/fact_session_results`;

In [0]:
# dbutils.fs.rm(
#     "s3://bizf1project123/Gold/Kalashri_increment/fact_session_results",
#     True
# )